# PRISM — BERTScore Evaluation (Google Colab)

Evaluate **RAG** and **non-RAG** answers against one or more reference-answer CSV files using **BERTScore (Precision, Recall, F1)**, **ROUGE-L (F1)**, and **BLEU**.

### Expected CSV files
- Candidate answers: `RAG_conversation_QA.csv`, `nRG_conversation_QA.csv`
- Reference answers: `gold_qa_gpt.csv`, `gold_qa_claude.csv`, `gold_qa_gemini.csv`, and/or `gold_qa_draft.csv`
- Each CSV should have `No`, `Question`, and `Answer` columns. Legacy reference CSVs with `question` and `reference` columns are also accepted.

### Procedure
1. Upload CSV files to the Google Colab **Files** sidebar (`/content`).
2. Run the cells and select **candidate** and **reference** files by number; enter `all` to select all listed files.
3. The notebook matches answers by `No` (and checks the question text), calculates scores for every candidate–reference pairing, and exports detailed and summary CSV files.

**Interpretation:** The scores measure agreement with the **selected reference answers**, not independent clinical or scientific correctness. Different reference LLMs may yield different scores. This notebook uses English (`lang="en"`) by default; the original `bert.ipynb` used Japanese (`lang="ja"`), so scores are not guaranteed to match earlier runs unless the original settings and reference answers are restored.

## 1. Install dependencies

In [ ]:
%pip -q install "bert-score>=0.3.13" "rouge-score>=0.1.2" "nltk>=3.8" "pandas>=2.0"

## 2. Find and select CSV files

In [ ]:
from pathlib import Path
import re
import pandas as pd
from IPython.display import display

BASE = Path('/content') if Path('/content').is_dir() else Path.cwd()
all_csv = sorted(BASE.glob('*.csv'))

candidate_files = [p for p in all_csv if p.name in {'RAG_conversation_QA.csv', 'nRG_conversation_QA.csv'}]
reference_files = [p for p in all_csv if p.name.startswith('gold_qa_') and p.name.endswith('.csv') and 'audit' not in p.name]

if not candidate_files:
    raise FileNotFoundError('Upload RAG_conversation_QA.csv and/or nRG_conversation_QA.csv.')
if not reference_files:
    raise FileNotFoundError('Upload at least one gold_qa_*.csv reference-answer file.')

def select_files(paths, label):
    print(f'\n{label}:')
    for i, p in enumerate(paths, 1):
        print(f'[{i}] {p.name}')
    while True:
        entry = input(f'Select {label} by number (comma-separated) or all: ').strip().lower()
        if entry == 'all':
            return paths
        try:
            indices = list(dict.fromkeys(int(x.strip()) for x in entry.split(',')))
            if indices and all(1 <= i <= len(paths) for i in indices):
                return [paths[i-1] for i in indices]
        except ValueError:
            pass
        print('Invalid selection. Enter e.g. 1,2 or all.')

selected_candidates = select_files(candidate_files, 'candidate files')
selected_references = select_files(reference_files, 'reference files')
print('\nSelected candidates:', [p.name for p in selected_candidates])
print('Selected references:', [p.name for p in selected_references])

## 3. Load and validate Q&A alignment

In [ ]:
def load_qa(path, is_reference=False):
    raw = pd.read_csv(path, encoding='utf-8-sig')
    colmap = {str(c).strip().lower(): c for c in raw.columns}
    question_col = colmap.get('question')
    answer_col = colmap.get('answer')
    if is_reference and answer_col is None:
        answer_col = colmap.get('reference')
    if question_col is None or answer_col is None:
        raise ValueError(f'{path.name}: expected Question and Answer (or reference) columns; found {list(raw.columns)}')
    df = pd.DataFrame({'Question': raw[question_col], 'Answer': raw[answer_col]})
    if 'no' in colmap:
        df['No'] = pd.to_numeric(raw[colmap['no']], errors='raise').astype(int)
    else:
        print(f'WARNING: {path.name} has no No column; using row order as question numbers.')
        df['No'] = range(1, len(df) + 1)
    if df['No'].duplicated().any():
        raise ValueError(f'{path.name}: duplicate question numbers')
    if df[['Question', 'Answer']].isna().any().any():
        raise ValueError(f'{path.name}: missing questions or answers')
    for col in ['Question', 'Answer']:
        df[col] = df[col].astype(str).str.strip().str.replace(r'^[QA]:\s*\*?', '', regex=True).str.strip()
    if (df[['Question', 'Answer']] == '').any().any():
        raise ValueError(f'{path.name}: blank questions or answers')
    return df[['No', 'Question', 'Answer']].sort_values('No').reset_index(drop=True)

candidate_data = {p.name: load_qa(p) for p in selected_candidates}
reference_data = {p.name: load_qa(p, is_reference=True) for p in selected_references}

pairs = []
for candidate_name, candidate in candidate_data.items():
    for reference_name, reference in reference_data.items():
        if set(candidate['No']) != set(reference['No']):
            raise ValueError(f'{candidate_name} vs {reference_name}: question numbers do not match')
        joined = candidate.merge(reference, on='No', suffixes=('_candidate', '_reference'), validate='one_to_one')
        normalize = lambda s: re.sub(r'\s+', ' ', s).strip().casefold()
        mismatch = joined['Question_candidate'].map(normalize) != joined['Question_reference'].map(normalize)
        if mismatch.any():
            bad = joined.loc[mismatch, 'No'].tolist()
            raise ValueError(f'{candidate_name} vs {reference_name}: question text mismatch at No {bad}. Check that all CSVs contain the same question set.')
        pairs.append((candidate_name, reference_name, joined))
        print(f'{candidate_name} vs {reference_name}: {len(joined)} matched questions')

## 4. Calculate BERTScore, ROUGE-L, and BLEU

In [ ]:
from bert_score import score as bert_score
from rouge_score import rouge_scorer
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

# English evaluation. For exact replication of an older run, use its original settings.
BERTSCORE_LANG = 'en'
BERTSCORE_MODEL_TYPE = None  # Set to the model used in the manuscript, if known.
BERTSCORE_RESCALE_WITH_BASELINE = False
BERTSCORE_IDF = False

rouge = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=False)
smooth = SmoothingFunction().method1
results = []

for candidate_name, reference_name, aligned in pairs:
    candidates = aligned['Answer_candidate'].tolist()
    references = aligned['Answer_reference'].tolist()
    kwargs = dict(lang=BERTSCORE_LANG, verbose=True,
                  rescale_with_baseline=BERTSCORE_RESCALE_WITH_BASELINE,
                  idf=BERTSCORE_IDF)
    if BERTSCORE_MODEL_TYPE is not None:
        kwargs['model_type'] = BERTSCORE_MODEL_TYPE
    precision, recall, f1 = bert_score(candidates, references, **kwargs)
    out = pd.DataFrame({
        'Candidate_File': candidate_name,
        'Reference_File': reference_name,
        'No': aligned['No'],
        'Question': aligned['Question_candidate'],
        'Reference': references,
        'Candidate': candidates,
        'BERTScore_Precision': precision.tolist(),
        'BERTScore_Recall': recall.tolist(),
        'BERTScore_F1': f1.tolist(),
        'ROUGE_L_F1': [rouge.score(ref, cand)['rougeL'].fmeasure for ref, cand in zip(references, candidates)],
        'BLEU': [sentence_bleu([ref.split()], cand.split(), smoothing_function=smooth)
                 for ref, cand in zip(references, candidates)],
    })
    results.append(out)
    print(f'{candidate_name} vs {reference_name}: mean BERTScore F1 = {out.BERTScore_F1.mean():.4f}')

details = pd.concat(results, ignore_index=True)
summary = (details.groupby(['Candidate_File', 'Reference_File'], as_index=False)
           .agg(N=('No', 'count'),
                BERTScore_Precision=('BERTScore_Precision', 'mean'),
                BERTScore_Recall=('BERTScore_Recall', 'mean'),
                BERTScore_F1=('BERTScore_F1', 'mean'),
                ROUGE_L_F1=('ROUGE_L_F1', 'mean'),
                BLEU=('BLEU', 'mean')))
display(summary)


## 5. Export results

In [ ]:
detail_path = BASE / 'prism_bertscore_detailed_results.csv'
summary_path = BASE / 'prism_bertscore_summary.csv'
details.to_csv(detail_path, index=False, encoding='utf-8-sig')
summary.to_csv(summary_path, index=False, encoding='utf-8-sig')
print('Saved:', detail_path)
print('Saved:', summary_path)
try:
    from google.colab import files
    files.download(str(detail_path))
    files.download(str(summary_path))
except ImportError:
    print('Files are available in the working directory.')